In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
import numpy as np

subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# SANITY CHECKS
# add idx colored plots to the ppt
# let's do a neuron by neuron trajectory plot too

# change dm structure
# beta weight sanity checks
# -> multiply beta weight by 1/binwidth_s and make sure identical to firing rate
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
)

encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
    strategy_filter="mf",
)

In [ ]:
encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
from core.viz import plot_scatter

plot_scatter(encoder.scores["encoder"], encoder_mb.scores["encoder"])
plot_scatter(encoder.scores["encoder"], encoder_mf.scores["encoder"])
plot_scatter(encoder_mb.scores["encoder"], encoder_mf.scores["encoder"])

In [ ]:
encoder.view_fits()

In [ ]:
from squiggs.renderers import WeightRendererTime
from squiggs.neuron_viewer import NeuronViewer
from core.data import tv_vals

r = WeightRendererTime(
    weights=encoder.encoder_weights,
    tv="response",
    weight_idxs=encoder.dm_idxs,
    tv_vals=tv_vals,
    tbin_centers=encoder.tbin_centers,
)

_ = NeuronViewer(num_units=encoder.num_units, render_func=r)

## the t-population

In [ ]:
# make the scatter but color by tpoint

In [ ]:
# select the neurons that lie along the axis
# plot their encoding for mb/mf
# is it just a different encoding pattern (i.e., they encode different things)
# or are they silent(er) in another strategy

## r2

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes
from utils.colors import colors_region

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
# scores
scores = {
    k: {
        f"{reg}": encoder_.scores[model][encoder_.reg_idxs[reg]]
        for reg in encoder_.regions
        for model in ["encoder"]
    }
    for k, encoder_ in encoders.items()
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
linewidths = {"baseline": 0.5, "encoder": 1}
styles = {
    f"{reg}": {
        "linestyle": linestyles[model],
        "linewidth": linewidths[model],
        "color": colors_region[reg],
    }
    for reg in encoder.regions
    for model in ["encoder"]
}

for k, scores_ in scores.items():
    plot_kdes(
        scores_,
        label=rf"$r^2$, {k}",
        xlim=(-0.25, 1),
        add_means=False,
        line_kwargs=styles,
    )

## weight comp between regions and strategies

### kde

In [ ]:
from core.viz import plot_kde_row

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

sc_mean = {
    k: {
        f"{reg}": encoder_.robs[:, :, encoder_.reg_idxs[reg]].mean(axis=(0, 1))
        * (1000 / encoder.binwidth_ms)
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

styles_reg = {reg: styles[f"{reg}, encoder"] for reg in encoder.regions}
_ = plot_kde_row(
    sc_mean, line_kwargs=styles_reg, label="avg fr (across trials & tbins)"
)

In [ ]:
encoder.fit_encoder()

In [ ]:
encoder.fit_encoder()
encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

In [ ]:
regressor = "response_right"
weights_strategy = {
    k: {
        f"{reg}": encoder_.encoder_weights[
            encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
        ]
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

In [ ]:
from core.viz import plot_kde_row
from utils.colors import colors_region, colors_strategy
from utils.paths import FIGURES_DIR

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

# styles
linewidths_strategy = {"full": 0.5, "mb": 1, "mf": 1}

styles_strategy = {f"{reg}": {"color": colors_region[reg]} for reg in encoder.regions}
styles_reg = {
    f"{k}": {
        "color": colors_strategy[k],
        "linewidth": linewidths_strategy[k],
    }
    for k in encoders
}

# iterate through all regressors and save
for regressor in encoder.dm_names:
    if "tents" not in regressor:
        # weights
        weights_strategy = {
            k: {
                f"{reg}": np.ravel(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for reg in encoder_.regions
            }
            for k, encoder_ in encoders.items()
        }

        weights_reg = {
            reg: {
                f"{k}": np.ravel(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for k, encoder_ in encoders.items()
            }
            for reg in encoder.regions
        }

        # plot
        for k, weights, styles in zip(
            ["strategy", "region"],
            [weights_strategy, weights_reg],
            [styles_strategy, styles_reg],
        ):
            fig, _ = plot_kde_row(
                weights, styles, title=rf"$\beta$ {regressor}", add_means=False
            )

            fpath = FIGURES_DIR / "time_resolved" / "bweight" / subj_id / sess_id
            fpath.mkdir(parents=True, exist_ok=True)
            fig.savefig(fpath / f"{regressor}_{k}.png", dpi=300, bbox_inches="tight")

### scatter, hist 2d, contour

In [ ]:
from core.data import tv_vals
from core.viz import plot_scatter, plot_hist2d, plot_2d_row
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for regr in encoder.tv_keys:
    norm_str = "norm" if encoder.norm else "nonorm"
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / norm_str

    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        print(regressor)
        weights = {
            reg: [
                np.concatenate(
                    encoder_.encoder_weights[
                        :, encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                    ]
                )
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        idxs = {
            reg: np.tile(range(encoder.num_bins), encoder.psths[reg].shape[0])
            for reg in encoder.regions
        }

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            color=idxs,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(
            fig,
            fpath / "scatter" / regressor / subj_id,
            f"{regressor}-{subj_id}_{sess_id}.png",
        )

        # hist2d
        fig, ax = plot_2d_row(
            plot_hist2d,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            sharey=True,
            sharex=True,
        )

        save_fig(
            fig,
            fpath / "hist2d" / regressor / subj_id,
            f"{regressor}-{subj_id}_{sess_id}.png",
        )

In [ ]:
len(weights["DLS"][1])

## across sessions

In [ ]:
from core.data import tv_vals

tv_vals

In [ ]:
from core.data import tv_vals, subject_ids, session_ids
from core.viz import plot_scatter, plot_2d_row
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

for subj_id in ["MR82", "MR83"]:
    print(subj_id)
    fpath = FIGURES_DIR / "time_resolved" / "bweight" / subj_id

    weights_master = {
        reg: {
            strategy: {
                f"{regr}_{val}": []
                for regr, vals in tv_vals.items()
                for val in vals
                if regr != "strategy"
            }
            for strategy in ["mb", "mf"]
        }
        for reg in ["DMS", "DLS"]
    }

    for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
        print(sess_id)
        try:
            encoder_mb = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                stepsize_s=0.1,
                strategy_filter="mb",
            )

            encoder_mf = make_tre(StrategyEncoder)(
                subj_id,
                sess_id,
                stepsize_s=0.1,
                strategy_filter="mf",
            )

            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()

        except ValueError:
            print("..too few trials")
            continue

        for reg, weights_reg in weights_master.items():
            for k in weights_reg["mb"].keys():
                print(k)
                try:
                    weights_reg["mb"][k].extend(
                        np.concatenate(
                            encoder_mb.encoder_weights[
                                :, encoder_mb.reg_idxs[reg], encoder_mb.dm_idxs[k]
                            ]
                        )
                    )
                    weights_reg["mf"][k].extend(
                        np.concatenate(
                            encoder_mf.encoder_weights[
                                :, encoder_mf.reg_idxs[reg], encoder_mf.dm_idxs[k]
                            ]
                        )
                    )
                except KeyError:
                    continue

    for regr in encoder.tv_keys:
        vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

        for val in vals:
            regressor = f"{regr}_{val}"
            weights = {
                reg: [weights_master[reg]["mb"], weights_master[reg]["mf"]]
                for reg in ["DMS", "DLS"]
            }

            # mn/mx
            mn = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).min()
            )
            mx = (
                1.05
                * np.concatenate(
                    [
                        weights_
                        for weights_strategy in weights.values()
                        for weights_ in weights_strategy
                    ]
                ).max()
            )

            # scatter
            fig, ax = plot_2d_row(
                plot_scatter,
                weights,
                mn=mn,
                mx=mx,
                xlabel="mb",
                ylabel="mf",
                title=rf"$\beta$ {regressor}",
                add_unity=True,
                add_lr=True,
            )

            save_fig(fig, fpath / "scatter", f"{regressor}.png")